# nanoAlphaZero — custom-game Colab

This notebook is a quick demonstration of nanoAlphaZero's game-agnostic logic.

Suppose you want to train AlphaZero on a completely new game like Tic-Tac-Toe but 4x4 (3 in a row wins).

Here's what you do

1. Create a PGX-styled env (ask an LLM)
2. Paste it below
3. Run the cells


In [ ]:
# Preserve Colab's JAX/libtpu pair, install runtime dependencies, then
# install the split package from the exact origin/eval commit inspected here.
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import jax

PACKAGE_REF = "cf95658735b285c57bdcfd1e1467b082c8f565ff"
print("JAX:", jax.__version__)
print("Devices:", jax.devices())
if not jax.devices() or jax.devices()[0].platform != "tpu":
    raise RuntimeError("No TPU found. Select a TPU runtime and restart the session.")

constraints = Path("/tmp/colab-jax-constraints.txt")
constraints.write_text(
    f"jax=={version('jax')}\n"
    f"jaxlib=={version('jaxlib')}\n"
)

runtime_dependencies = [
    "pgx1 @ git+https://github.com/wtedw/pgx1.git@fa313c84338d93ab96fc02bc7c658364bf43098f",
    "mctx @ git+https://github.com/wtedw/mctx.git@6cf1a39",
    "flashbax @ git+https://github.com/instadeepai/flashbax.git@e0199d7bb232c622a19d3c28f9d6b34eb8215eab",
    "flax==0.10.1",
    "optax==0.2.7",
    "chex==0.1.91",
    "safetensors==0.8.0",
    "wandb==0.21.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--constraint", str(constraints), *runtime_dependencies],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        f"nanoalphazero @ git+https://github.com/wtedw/nanoAlphaZero.git@{PACKAGE_REF}",
    ],
    check=True,
)
print("Installed nanoAlphaZero package ref:", PACKAGE_REF[:8])


## Imports


In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np

import nanoalphazero.core as az_core
import nanoalphazero.play as az_play
from nanoalphazero.checkpoint import load_checkpoint, save_checkpoint
from nanoalphazero.config import get_ttt_config
from nanoalphazero.core import make_alphazero


## Custom N×NxK game


In [ ]:
# Copyright 2023 The Pgx Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Generalized N x N Tic-Tac-Toe used as the custom-environment example."""

import dataclasses
from typing import NamedTuple, Optional

import jax
import jax.numpy as jnp
from jax import Array, lax

TRUE = jnp.bool_(True)
FALSE = jnp.bool_(False)


class GameState(NamedTuple):
    color: Array
    board: Array
    winner: Array


def _make_winning_lines(n: int, k: int) -> Array:
    """Return shape [num_lines, k] containing flattened board indices."""
    lines = []
    for r in range(n):
        for c in range(n - k + 1):
            lines.append([r * n + c + i for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n):
            lines.append([(r + i) * n + c for i in range(k)])
    for r in range(n - k + 1):
        for c in range(n - k + 1):
            lines.append([(r + i) * n + c + i for i in range(k)])
    for r in range(n - k + 1):
        for c in range(k - 1, n):
            lines.append([(r + i) * n + c - i for i in range(k)])
    return jnp.asarray(lines, dtype=jnp.int32)


class Game:
    def __init__(self, n: int, k: int = 3):
        if n < 1:
            raise ValueError(f"n must be >= 1, got {n}")
        if k < 1:
            raise ValueError(f"k must be >= 1, got {k}")
        if k > n:
            raise ValueError(f"k ({k}) cannot be larger than n ({n})")
        self.n = n
        self.k = k
        self._winning_lines = _make_winning_lines(n, k)

    def init(self) -> GameState:
        return GameState(
            color=jnp.int32(0),
            board=-jnp.ones(self.n * self.n, dtype=jnp.int32),
            winner=jnp.int32(-1),
        )

    def step(self, state: GameState, action: Array) -> GameState:
        board = state.board.at[action].set(state.color)
        won = (board[self._winning_lines] == state.color).all(axis=1).any()
        winner = lax.select(won, state.color, jnp.int32(-1))
        return state._replace(
            board=board,
            color=(state.color + 1) % 2,
            winner=winner,
        )

    def observe(self, state: GameState, color: Optional[Array] = None) -> Array:
        if color is None:
            color = state.color
        grid = state.board.reshape((self.n, self.n))
        return jnp.stack(
            [
                grid == color,
                grid == (1 - color),
                jnp.full((self.n, self.n), color, dtype=jnp.bool_),
                jnp.ones((self.n, self.n), dtype=jnp.bool_),
            ],
            axis=-1,
        )

    def legal_action_mask(self, state: GameState) -> Array:
        return state.board < 0

    def is_terminal(self, state: GameState) -> Array:
        return (state.winner >= 0) | jnp.all(state.board != -1)

    def rewards(self, state: GameState) -> Array:
        return lax.select(
            state.winner >= 0,
            jnp.float32([-1, -1]).at[state.winner].set(1),
            jnp.zeros(2, dtype=jnp.float32),
        )


def _field(factory):
    return dataclasses.field(default_factory=factory)


@jax.tree_util.register_dataclass
@dataclasses.dataclass(frozen=True)
class State:
    current_player: Array = _field(lambda: jnp.int32(0))
    observation: Array = _field(
        lambda: jnp.zeros((1, 1, 4), dtype=jnp.bool_)
    )
    rewards: Array = _field(lambda: jnp.zeros(2, dtype=jnp.float32))
    terminated: Array = _field(lambda: FALSE)
    truncated: Array = _field(lambda: FALSE)
    legal_action_mask: Array = _field(lambda: jnp.ones(1, dtype=jnp.bool_))
    _step_count: Array = _field(lambda: jnp.int32(0))
    _x: GameState = _field(
        lambda: GameState(
            color=jnp.int32(0),
            board=-jnp.ones(1, dtype=jnp.int32),
            winner=jnp.int32(-1),
        )
    )

    def replace(self, **kwargs) -> "State":
        return dataclasses.replace(self, **kwargs)

    @property
    def env_id(self) -> str:
        return "custom_tic_tac_toe"


class TicTacToeGeneral:
    """N x N Tic-Tac-Toe with k consecutive pieces required to win."""

    def __init__(self, n: int = 3, k: int = 3):
        self.n = n
        self.k = k
        self._game = Game(n=n, k=k)

    def init(self, key: Optional[Array] = None) -> State:
        del key
        x = self._game.init()
        return State(
            current_player=jnp.int32(0),
            observation=self._game.observe(x),
            legal_action_mask=self._game.legal_action_mask(x),
            _x=x,
        )

    def step(
        self,
        state: State,
        action: Array,
        key: Optional[Array] = None,
    ) -> State:
        del key
        is_illegal = ~self._check_legality(state, action)
        current_player = state.current_player
        state = lax.cond(
            state.terminated | state.truncated,
            lambda: state.replace(rewards=jnp.zeros_like(state.rewards)),
            lambda: self._step(
                state.replace(_step_count=state._step_count + 1), action
            ),
        )
        state = lax.cond(
            is_illegal,
            lambda: self._step_with_illegal_action(state, current_player),
            lambda: state,
        )
        return lax.cond(
            state.terminated,
            lambda: state.replace(
                legal_action_mask=jnp.ones_like(state.legal_action_mask)
            ),
            lambda: state,
        )

    def observe(
        self,
        state: State,
        player_id: Optional[Array] = None,
    ) -> Array:
        if player_id is None:
            player_id = state.current_player
        curr_color = state._x.color
        my_color = lax.select(
            player_id == state.current_player,
            curr_color,
            1 - curr_color,
        )
        return lax.stop_gradient(self._game.observe(state._x, my_color))

    def _step(self, state: State, action: Array) -> State:
        x = self._game.step(state._x, action)
        state = state.replace(
            current_player=(state.current_player + 1) % 2,
            _x=x,
        )
        terminated = self._game.is_terminal(x)
        rewards = self._game.rewards(x)
        rewards = lax.select(
            state.current_player != x.color,
            jnp.flip(rewards),
            rewards,
        )
        rewards = lax.select(
            terminated,
            rewards,
            jnp.zeros(2, dtype=jnp.float32),
        )
        return state.replace(
            observation=self.observe(state, state.current_player),
            legal_action_mask=self._game.legal_action_mask(x),
            rewards=rewards,
            terminated=terminated,
        )

    def _check_legality(self, state: State, action: Array) -> Array:
        mask_i32 = state.legal_action_mask.astype(jnp.int32)
        one_hot_a = jax.nn.one_hot(
            action,
            mask_i32.shape[0],
            dtype=jnp.int32,
        )
        return jnp.dot(one_hot_a, mask_i32).astype(jnp.bool_)

    def _step_with_illegal_action(self, state: State, loser: Array) -> State:
        rewards = jnp.where(
            jnp.arange(2) == loser,
            -1.0,
            1.0,
        ).astype(jnp.float32)
        return state.replace(rewards=rewards, terminated=TRUE)

    @property
    def id(self) -> str:
        return "custom_tic_tac_toe"

    @property
    def version(self) -> str:
        return "v0"

    @property
    def num_players(self) -> int:
        return 2

    @property
    def num_actions(self) -> int:
        return self.n * self.n


## Notebook-local environment registry adapter

nanoAlphaZero doesn't have an API to support new environments, so we have to hack `core.make_env`.


In [ ]:
ENV_REGISTRY = {}
_BUILTIN_MAKE_ENV = az_core.make_env


def register_env(env_id, factory, *, replace=False):
    if not isinstance(env_id, str) or not env_id.strip():
        raise ValueError("env_id must be a non-empty string")
    if not callable(factory):
        raise TypeError("factory must be callable")
    if env_id in ENV_REGISTRY and not replace:
        raise ValueError(f"{env_id!r} is already registered")
    ENV_REGISTRY[env_id] = factory


def make_registered_env(config):
    env_id = config["env_id"]
    if env_id not in ENV_REGISTRY:
        return _BUILTIN_MAKE_ENV(config)

    env = ENV_REGISTRY[env_id]()
    required = ("init", "step", "observe", "num_actions")
    missing = [name for name in required if not hasattr(env, name)]
    if missing:
        raise TypeError(f"Custom environment is missing: {', '.join(missing)}")

    step = env.step
    auto_step = az_core.auto_reset(step, env.init)
    batched_init = jax.jit(jax.vmap(env.init))
    batched_step = jax.jit(jax.vmap(step))
    batched_auto_step = jax.jit(jax.vmap(auto_step))
    batched_observe = jax.jit(jax.vmap(env.observe))
    single_state = env.init(jax.random.PRNGKey(0))

    def init_dummy_estate(batch_size):
        return batched_init(jax.random.split(jax.random.PRNGKey(0), batch_size))

    probe = batched_init(jax.random.split(jax.random.PRNGKey(42), 1))
    observation = batched_observe(probe, probe.current_player)
    return az_core.WrappedEnv(
        obs_shape=observation.shape[1:],
        num_actions=env.num_actions,
        init=batched_init,
        step=batched_step,
        autostep=batched_auto_step,
        init_dummy_estate=init_dummy_estate,
        single_estate=single_state,
        observe=batched_observe,
        replay_batch=getattr(env, "replay_batch", None),
    )


# core.make_alphazero looks up az_core.make_env at runtime. play.py imported its
# own reference, so install the same resolver there as well.
az_core.make_env = make_registered_env
az_play.make_env = make_registered_env


## Register the game and build its config

Change only `BOARD_SIZE` and `WIN_LENGTH`. Rerun this cell after a runtime reset.
All shape-dependent search, buffer, and exploration values are derived here.


In [ ]:
BOARD_SIZE = 4
WIN_LENGTH = 3
CUSTOM_ENV_ID = f"custom_ttt_{BOARD_SIZE}x{BOARD_SIZE}_k{WIN_LENGTH}"
CUSTOM_ENV = lambda: TicTacToeGeneral(n=BOARD_SIZE, k=WIN_LENGTH)
register_env(CUSTOM_ENV_ID, CUSTOM_ENV, replace=True)


def custom_config():
    n, k = BOARD_SIZE, WIN_LENGTH
    if n < 4 or not 1 <= k <= n:
        raise ValueError("Require BOARD_SIZE >= 4 and 1 <= WIN_LENGTH <= BOARD_SIZE")
    game_max_steps = n * n
    root_actions = min(16, game_max_steps)
    survivors = max(1, root_actions // 2)
    # Use the basic ttt config
    config = get_ttt_config()
    batch_size = config["selfplay_batch_size"]
    selfplay_buffer_len = game_max_steps + 10
    replay_buffer_len = config["replay_buffer_total_size"] // batch_size
    warmup_steps = selfplay_buffer_len + replay_buffer_len
    config.update(
        env_id=CUSTOM_ENV_ID,
        game_name=CUSTOM_ENV_ID,
        boardsize=n,
        game_max_steps=game_max_steps,
        game_obs_shape=None,
        game_num_actions=None,
        num_iters=config["num_iters"] // 2,
        num_exploratory_moves=max(1, game_max_steps // 2),
        mcts_num_simulations=root_actions + survivors,  # this field is unsupported at the moment
        mcts_max_m=root_actions,
        mcts_num_root_considered=8,
        mcts_num_survivors=4,
        mcts_num_k_actions=game_max_steps,
        lr_warmup_steps=warmup_steps,
        replay_buffer_warmup_steps=warmup_steps,
        selfplay_buffer_min_len=game_max_steps,
        selfplay_buffer_max_len=game_max_steps,
        enable_wandb=False,
    )
    return config


CONFIG = custom_config()


## Smoke test

This confirms registry resolution and the package wrapper before model/buffer
allocation. PGX auto-reset requires one PRNG key per batched game.


In [ ]:
wenv = az_core.make_env(CONFIG)
keys = jax.random.split(jax.random.PRNGKey(0), 2)
state = wenv.init(keys)
observation = wenv.observe(state, state.current_player)
actions = jnp.argmax(state.legal_action_mask, axis=1).astype(jnp.int32)
step_keys = jax.random.split(jax.random.PRNGKey(1), 2)
next_state = wenv.autostep(state, actions, step_keys)

assert observation.shape == (2, BOARD_SIZE, BOARD_SIZE, 4)
assert state.legal_action_mask.shape == (2, BOARD_SIZE * BOARD_SIZE)
assert next_state.rewards.shape == (2, 2)
print("Custom package environment OK:", wenv)


## Empty-board diagnostics

This reports the model's P1-to-move value, W/D/L probabilities, and N×N policy
logits. A confident forced P1 win should approach value `+1` and win probability
`1`. The training loop calls this periodically without rebuilding the model.


In [ ]:
def print_initial_evaluation(az, runner_state):
    state = az.env.init_dummy_estate(batch_size=1)
    obs = az.env.observe(state, state.current_player)
    logits, value, wdl_logits = runner_state.model_ts.apply_fn(
        {"params": runner_state.model_ts.params},
        obs,
        state.legal_action_mask,
        deterministic=True,
        return_wdl_logits=True,
    )
    board_size = az.config["boardsize"]
    logits = np.asarray(logits[0]).reshape((board_size, board_size))
    wdl = np.asarray(jax.nn.softmax(wdl_logits[0]))
    print(
        f"P1 initial value={float(value[0]):+.3f} | "
        f"win={wdl[0]:.3f} draw={wdl[2]:.3f} loss={wdl[1]:.3f}"
    )
    print("Policy logits:")
    for row in logits:
        print("  " + "  ".join(f"{x:+.2f}" for x in row))


## AlphaZero training loop

`az.run_fn` is just one compiled cycle containing the three expensive phases:

1. self-play,
2. drain (move completed-games into replay buffer),
3. train (perform gradient update).

Diagnostics run after cycle 1 and then every `CONFIG["diagnostic_period"]` cycles.
The custom config halves the baseline Tic-Tac-Toe `num_iters`. Each warmup and
training line reports the synchronized wall time of its `run_fn` call.


In [ ]:
SAVE_PATH = f"artifacts/alphazero_{CUSTOM_ENV_ID}.safetensors"
MAX_TRAINING_CYCLES = None  # e.g. 5 for a short trial
DIAGNOSTIC_PERIOD = CONFIG.get("diagnostic_period", 100)

rng = jax.random.PRNGKey(42)
rng, build_key = jax.random.split(rng)
az = make_alphazero(CONFIG, build_key)
runner_state = az.runner_state

# 1. Fill replay memory while the optimizer is frozen.
warmup_cycles = (
    az.config["replay_buffer_warmup_steps"] // az.config["cycle_n_selfplay"]
)
print(f"Warmup: {warmup_cycles} cycles")
print("(this will take some time as JAX JIT compiles the first call)")
for cycle in range(warmup_cycles):
    run_fn_started = time.time()
    runner_state, _ = az.run_fn(runner_state, jnp.array(True))
    runner_state.model_ts.step.block_until_ready()
    run_fn_seconds = time.time() - run_fn_started
    print(
        f"  warmup {cycle + 1}/{warmup_cycles} | "
        f"run_fn={run_fn_seconds:.2f}s"
    )

# 2. Alternate self-play, replay drain, and gradient training.
training_cycles = az.config["num_iters"] // az.config["cycle_n_selfplay"]
if MAX_TRAINING_CYCLES is not None:
    training_cycles = min(training_cycles, MAX_TRAINING_CYCLES)

print(
    f"Training: {training_cycles} cycles "
    f"({az.config['num_iters']} total iterations)"
)
for cycle in range(1, training_cycles + 1):
    run_fn_started = time.time()
    runner_state, (scalar_metrics, _) = az.run_fn(
        runner_state, jnp.array(False)
    )
    jax.tree_util.tree_map(lambda x: x.block_until_ready(), runner_state)
    run_fn_seconds = time.time() - run_fn_started
    last = jax.tree_util.tree_map(lambda x: float(x[-1]), scalar_metrics)
    print(
        f"cycle {cycle}/{training_cycles} | run_fn={run_fn_seconds:.2f}s | "
        f"loss={last['total_loss']:.4f} "
        f"value={last['loss_v']:.4f} policy={last['loss_pi']:.4f}"
    )

    if cycle == 1 or (
        DIAGNOSTIC_PERIOD and cycle % DIAGNOSTIC_PERIOD == 0
    ):
        print_initial_evaluation(az, runner_state)

save_checkpoint(runner_state.model_ts.params, az.config, SAVE_PATH)
print("Saved:", SAVE_PATH)


## Optional: play against the model

The package play path automatically uses the same registered environment because
`az_play.make_env` was installed by the adapter cell.


In [ ]:
# az_play.play_against_model(
#     CONFIG,
#     runner_state.model_ts.params,
#     human_player=0,
#     num_simulations=None,
# )

# In a fresh session, rerun through registration, then load and play:
# params, model_config = load_checkpoint(SAVE_PATH)
# az_play.play_against_model(CONFIG, params, human_player=0)
